# Two-boundary tuning and constrained XGBoost
This notebook reads real saved results; no training or holdout inference is repeated.

In [1]:
import json
from pathlib import Path
EXPERIMENT=Path.cwd() if (Path.cwd()/'protocol.json').exists() else globals().get('EXPERIMENT')
assert EXPERIMENT is not None, 'Set EXPERIMENT to the saved experiment folder.'
load=lambda n:json.loads((EXPERIMENT/n).read_text())
print(load('protocol.json')['objective'])

Maximize mean outer-fold accuracy among candidates satisfying BOTH high and sometimes recalls, macro-F1 and QWK vs refitted baseline on EVERY scoring fold. High floor max(60%,baseline); sometimes floor baseline. Thresholds chosen on inner calibration with same constraints. No thresholds fitted on scoring fold.


## Decision rule
High if P(high) ≥ h; otherwise sometimes if P(sometimes)/(P(rarely)+P(sometimes)) ≥ s; otherwise rarely. Thresholds do not alter the reported probabilities. Both class recalls are constrained.

In [2]:
f=load('frozen_selection.json')
for name,policies in f['candidates'].items():
    print(name)
    for model,row in policies.items(): print(model,row['thresholds'],'validation accuracy:',row['metrics']['accuracy'],'recalls:',row['metrics']['recall'],'feasible:',row['feasible'])

boundaries_only
xgb {'high': 0.27792591452598575, 'middle': 0.4678294798942472} validation accuracy: 0.663617819306442 recalls: [0.8102394549644324, 0.2345068373581612, 0.6700393479482856] feasible: True
ensemble {'high': 0.33022436197988114, 'middle': 0.5001174221478011} validation accuracy: 0.6844113969862473 recalls: [0.852920549043182, 0.23858015711376201, 0.6003372681281619] feasible: True
tuned_xgb
xgb {'high': 0.27831292152404785, 'middle': 0.4676314440168318} validation accuracy: 0.6633546094623939 recalls: [0.8088367899008115, 0.23916205993599068, 0.6666666666666666] feasible: False
ensemble {'high': 0.32571284766081937, 'middle': 0.4817060304776567} validation accuracy: 0.6778969533460552 recalls: [0.8354874261096082, 0.2598196101251091, 0.6014614952220348] feasible: True


## Leakage controls
Every fold refits its own preprocessing and baseline. Training, early stopping, threshold calibration and scoring have disjoint profile groups. The score fold does not choose its own thresholds.

In [3]:
import csv
from collections import Counter
roles=list(csv.DictReader((EXPERIMENT/'cv_roles.csv').open()))
print(Counter((r['fold'],r['role']) for r in roles))
print('Trials:',len(list((EXPERIMENT/'trials').glob('*.json'))))

Counter({('1', 'score'): 64125, ('0', 'score'): 64123, ('1', 'fit'): 44891, ('0', 'fit'): 44880, ('0', 'threshold_calibration'): 12828, ('1', 'threshold_calibration'): 12822, ('0', 'early_stop'): 6417, ('1', 'early_stop'): 6410})
Trials: 40


## Constrained search
We maximize accuracy only among configurations satisfying the recall, macro-F1 and QWK conditions in both outer folds. The default baseline is an explicit trial, with original features and fixed training settings.

In [4]:
c=load('cv_selection.json')
print('Feasible trials:',c['feasible_trials'])
print('Diagnostic only:',c['diagnostic_only'])
print(json.dumps(c['candidate']['params'],indent=2))

Feasible trials: 1
Diagnostic only: False
{
  "n_estimators": 350,
  "max_depth": 5,
  "learning_rate": 0.04,
  "min_child_weight": 12,
  "reg_lambda": 5,
  "reg_alpha": 0,
  "subsample": 0.85,
  "colsample_bytree": 0.85,
  "gamma": 0,
  "weight_power": 0.5,
  "middle_weight": 1.0,
  "high_weight": 1.0,
  "mode": "original"
}


## Frozen final evaluation
This internal holdout has been reused historically. Treat results as internal evidence, not a claim of external generalization.

In [5]:
t=load('test_report.json')
for model in ['xgb','ensemble']:
    for version in ['original','candidate']:
        m=t[model][version]
        print(model,version,'accuracy',round(m['accuracy'],4),'high recall',round(m['recall'][2],4),'sometimes recall',round(m['recall'][1],4),'high missed',m['high_missed'])
print('Always rarely accuracy:',t['majority_accuracy'],'both minority recalls: 0')
print('Approved for runtime:',t['approved_for_runtime'])

xgb original accuracy 0.6754 high recall 0.6533 sometimes recall 0.2095 high missed 1255
xgb candidate accuracy 0.6647 high recall 0.6685 sometimes recall 0.237 high missed 1200
ensemble original accuracy 0.6839 high recall 0.6014 sometimes recall 0.2268 high missed 1443
ensemble candidate accuracy 0.6839 high recall 0.6014 sometimes recall 0.2265 high missed 1443
Always rarely accuracy: 0.6578227749202422 both minority recalls: 0
Approved for runtime: False


In [6]:
print(json.dumps(load('gate_reasons.json'),indent=2))

{
  "xgb": [
    "Frozen candidate did not qualify on validation; diagnostic only.",
    "Accuracy did not improve.",
    "qwk decreased.",
    "Paired 95% interval does not establish a positive accuracy gain."
  ],
  "ensemble": [
    "Frozen candidate did not qualify on validation; diagnostic only.",
    "Sometimes recall decreased.",
    "macro_f1 decreased.",
    "Paired 95% interval does not establish a positive accuracy gain."
  ]
}
